In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("~/data/legal/cour_tj_correspondence.csv")

In [3]:
", ".join(list(df[df.columns[1]].unique()))

"Cour d'Appel de Lyon, Cour d'Appel d'Amiens, Cour d'Appel de Riom, Cour d'Appel d'Aix-en-Provence, Cour d'Appel de Grenoble, Cour d'Appel de Nîmes, Cour d'Appel de Reims, Cour d'Appel de Toulouse, Cour d'Appel de Montpellier, Cour d'Appel de Caen, Cour d'Appel de Bordeaux, Cour d'Appel de Poitiers, Cour d'Appel de Bourges, Cour d'Appel de Limoges, Cour d'Appel de Dijon, Cour d'Appel de Rennes, Cour d'Appel de Besançon, Cour d'Appel de Rouen, Cour d'Appel de Versailles, Cour d'Appel de Bastia, Cour d'Appel d'Agen, Cour d'Appel d'Orléans, Cour d'Appel de Pau, Cour d'Appel d'Angers, Cour d'Appel de Nancy, Cour d'Appel de Metz, Cour d'Appel de Douai, Cour d'Appel de Colmar, Cour d'Appel de Chambéry, Cour d'Appel de Paris, Cour d'Appel de Basse-Terre, Cour d'Appel de Fort-de-France, Cour d'appel de Cayenne, Cour d'Appel de Saint-Denis-de-La Réunion, Tribunal Supérieur d'Appel de Saint-Pierre-et-Miquelon, Cour d'Appel de Nouméa, Cour d'Appel de Papeete"

In [4]:
sorted((df[df.columns[1]].unique()))

["Cour d'Appel d'Agen",
 "Cour d'Appel d'Aix-en-Provence",
 "Cour d'Appel d'Amiens",
 "Cour d'Appel d'Angers",
 "Cour d'Appel d'Orléans",
 "Cour d'Appel de Basse-Terre",
 "Cour d'Appel de Bastia",
 "Cour d'Appel de Besançon",
 "Cour d'Appel de Bordeaux",
 "Cour d'Appel de Bourges",
 "Cour d'Appel de Caen",
 "Cour d'Appel de Chambéry",
 "Cour d'Appel de Colmar",
 "Cour d'Appel de Dijon",
 "Cour d'Appel de Douai",
 "Cour d'Appel de Fort-de-France",
 "Cour d'Appel de Grenoble",
 "Cour d'Appel de Limoges",
 "Cour d'Appel de Lyon",
 "Cour d'Appel de Metz",
 "Cour d'Appel de Montpellier",
 "Cour d'Appel de Nancy",
 "Cour d'Appel de Nouméa",
 "Cour d'Appel de Nîmes",
 "Cour d'Appel de Papeete",
 "Cour d'Appel de Paris",
 "Cour d'Appel de Pau",
 "Cour d'Appel de Poitiers",
 "Cour d'Appel de Reims",
 "Cour d'Appel de Rennes",
 "Cour d'Appel de Riom",
 "Cour d'Appel de Rouen",
 "Cour d'Appel de Saint-Denis-de-La Réunion",
 "Cour d'Appel de Toulouse",
 "Cour d'Appel de Versailles",
 "Cour d'appel

In [5]:
len(list(df[df.columns[1]].unique()))

37

In [6]:
dfm = pd.read_csv("/home/alexander/data/legal/justice.dataset.c/mentions.csv")

In [7]:
dfm.head()

,filename,position,mention
0,pourvoi_n°22-80.124_04_01_2023.pdf,336,cour d'appel de Montpellier
1,pourvoi_n°22-80.393_04_01_2023.pdf,7538,cour d'appel de Versailles
2,pourvoi_n°22-80.393_04_01_2023.pdf,7785,cour d'appel de Versailles
3,pourvoi_n°22-80.696_04_01_2023.pdf,392,cour d'appel de Riom
4,pourvoi_n°22-80.696_04_01_2023.pdf,4999,cour d'appel de Riom


In [8]:
dfmr = dfm.drop_duplicates("filename", keep="first")

In [9]:
ca_dfmr = set(dfmr.mention.unique())

In [10]:
len(ca_dfmr)

43

In [11]:
ca_dfg = set(df[df.columns[1]].unique())

In [12]:
len(ca_dfg & ca_dfmr)

0

In [13]:
# ca_dfmr, ca_dfg

In [61]:
import unicodedata
from rapidfuzz import fuzz


def normalize_text(text):
    """
    Normalize text by:
    - Converting to lowercase
    - Removing accents
    - Standardizing apostrophes
    - Removing extra whitespace
    """
    # Convert to lowercase
    text = text.lower()

    # Remove accents
    text = "".join(
        c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn"
    )

    # Standardize apostrophes
    text = text.replace("'", "'").replace("'", "'")

    # Remove extra whitespace
    text = " ".join(text.split())

    return text


def extract_city_name(text):
    """
    Extract the city name from a court location string.
    Assumes format: "cour d'appel de [CITY]" or similar
    """
    normalized = normalize_text(text)
    # Remove common prefixes
    prefixes = [
        "cour d'appel de",
        "cour d'appel d'",
        "cour d appel de",
        "cour d appel d",
    ]
    for prefix in prefixes:
        if normalized.startswith(prefix):
            return normalized[len(prefix) :].strip()
    return normalized


def create_location_mapping(set1, set2, threshold=80, method="token_sort_ratio"):
    """
    Create a mapping between two sets of geographic locations using fuzzy matching.

    Parameters:
    -----------
    set1 : set or list
        First set of locations
    set2 : set or list
        Second set of locations
    threshold : int, default=80
        Minimum similarity score (0-100) to consider a match
    method : str, default='token_sort_ratio'
        Fuzzy matching method: 'ratio', 'partial_ratio', 'token_sort_ratio', 'token_set_ratio'

    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: 'set1_original', 'set2_original', 'set1_normalized',
        'set2_normalized', 'similarity_score'
    """
    # Convert to lists if sets
    list1 = list(set1)
    list2 = list(set2)

    # Normalize both sets
    normalized1 = {loc: normalize_text(loc) for loc in list1}
    normalized2 = {loc: normalize_text(loc) for loc in list2}

    # Extract city names for better matching
    cities1 = {loc: extract_city_name(loc) for loc in list1}
    cities2 = {loc: extract_city_name(loc) for loc in list2}

    # Choose scorer based on method
    scorer_map = {
        "ratio": fuzz.ratio,
        "partial_ratio": fuzz.partial_ratio,
        "token_sort_ratio": fuzz.token_sort_ratio,
        "token_set_ratio": fuzz.token_set_ratio,
    }
    scorer = scorer_map.get(method, fuzz.token_sort_ratio)

    # Create mappings
    mappings = []

    for loc1_orig in list1:
        city1 = cities1[loc1_orig]

        # Find all candidates with their scores
        candidates = []
        for loc2_orig in list2:
            city2 = cities2[loc2_orig]

            # Check if city1 is a prefix of city2 (handles partial names like "Saint" -> "Saint-Denis")
            if city2.startswith(city1) or city1.startswith(city2):
                # Boost score for prefix matches
                base_score = scorer(city1, city2)
                # Give significant boost if one is a prefix of the other
                prefix_boost = (
                    20 if len(city1) > 3 else 10
                )  # More boost for longer prefixes
                score = min(100, base_score + prefix_boost)
            else:
                # Regular fuzzy matching
                score = scorer(city1, city2)

            if score >= threshold:
                candidates.append((loc2_orig, city2, score))

        if candidates:
            # Sort by score (descending), then by length difference (prefer longer matches for partial names)
            candidates.sort(
                key=lambda x: (x[2], -abs(len(city1) - len(x[1]))), reverse=True
            )

            # Take the best match
            loc2_orig, matched_city, score = candidates[0]

            mappings.append(
                {
                    "set1_original": loc1_orig,
                    "set2_original": loc2_orig,
                    "set1_normalized": normalized1[loc1_orig],
                    "set2_normalized": normalized2[loc2_orig],
                    "similarity_score": score,
                }
            )

    # Create DataFrame
    df = pd.DataFrame(mappings)

    # Sort by similarity score (descending)
    if not df.empty:
        df = df.sort_values("similarity_score", ascending=False).reset_index(drop=True)

    return df

In [62]:
mapping_df = create_location_mapping(ca_dfmr, ca_dfg, threshold=70)

In [63]:
mapping_df.sort_values("similarity_score", ascending=True).to_csv(
    "/home/alexander/tmp/cour.map.csv"
)

In [64]:
mapping_df.sort_values("similarity_score", ascending=True)

,set1_original,set2_original,set1_normalized,set2_normalized,similarity_score
42,cour d'appel de SaintDenis de La,Cour d'Appel de Saint-Denis-de-La Réunion,cour d'appel de saintdenis de la,cour d'appel de saint-denis-de-la reunion,76.712329
41,cour d'appel de Saint-Denis de La,Cour d'Appel de Saint-Denis-de-La Réunion,cour d'appel de saint-denis de la,cour d'appel de saint-denis-de-la reunion,78.378378
40,cour d'appel de Saint-Denis,Cour d'Appel de Saint-Denis-de-La Réunion,cour d'appel de saint-denis,cour d'appel de saint-denis-de-la reunion,79.411765
39,cour d'appel de Saint,Cour d'Appel de Douai,cour d'appel de saint,cour d'appel de douai,85.714286
38,tribunal supérieur d'appel de Saint-Pierre,Tribunal Supérieur d'Appel de Saint-Pierre-et-...,tribunal superieur d'appel de saint-pierre,tribunal superieur d'appel de saint-pierre-et-...,87.500000
37,cour d'appel de BasseTerre,Cour d'Appel de Basse-Terre,cour d'appel de basseterre,cour d'appel de basse-terre,98.113208
36,cour d'appel de Fort-deFrance,Cour d'Appel de Fort-de-France,cour d'appel de fort-defrance,cour d'appel de fort-de-france,98.305085
0,Cour d'appel de Nancy,Cour d'Appel de Nancy,cour d'appel de nancy,cour d'appel de nancy,100.000000
8,cour d'appel de Rennes,Cour d'Appel de Rennes,cour d'appel de rennes,cour d'appel de rennes,100.000000
1,cour d'appel de Paris,Cour d'Appel de Paris,cour d'appel de paris,cour d'appel de paris,100.000000
